# Milestone 1: Automated Pipeline for arXiv Metadata, TeX Sources, and References

Notebook này phục vụ việc thực thi trọn vẹn toàn bộ quy trình Milestone 1 cho từng bài báo trong khoảng `start_id` đến `end_id`:
1. **Cào Metadata** từ arXiv API -> lưu `data/papers.jsonl` và ghi thống kê **Bảng 1: `m1_discovery_stats.csv`**.
2. **Tải mã nguồn & giải nén** chỉ giữ lại các file `.tex` và `.bib` (loại bỏ hoàn toàn các file ảnh/hình nhị phân, `.sty`, `.cls`, `.pdf`) -> lưu vào `data/raw/<clean_arxiv_id>/` và ghi thống kê **Bảng 2: `m1_download_stats.csv`**.
3. **Cào danh sách trích dẫn (References)** từ Semantic Scholar Graph API -> lưu `data/references.jsonl` và ghi thống kê **Bảng 3: `m1_reference_stats.csv`**.
4. **Đo lường Benchmark Hệ thống** tự động trong suốt quá trình chạy -> ghi thống kê **Bảng 4: `m1_system_benchmark.csv`**.

### Cơ chế chống lỗi mạng & quá tải (Rate limit & Retry):
- Tuân thủ khoảng cách tối thiểu giữa các request: $\ge 3.0$s với arXiv, $\ge 1.0$s với Semantic Scholar.
- Tự động retry với exponential backoff kèm jitter ngẫu nhiên và thời gian trần (ceiling) khi gặp lỗi HTTP 429 (Too Many Requests), HTTP 500/502/503/504, và sự cố gián đoạn kết nối mạng (do S2 API không trả về header Retry-After).
- Fail fast không retry với HTTP 404.
- Mỗi request lỗi tạm thời được gọi tối đa **5 lần tổng cộng**; hết lượt thì ghi lỗi và chuyển sang ID tiếp theo.
- Hỗ trợ Resume: bỏ qua kết quả hoàn tất còn hợp lệ; lỗi tạm thời được thử lại ở lần chạy sau khi hết thời gian chờ (60 giây, tăng đôi, tối đa 1 giờ).
- Trạng thái của từng bước lưu trong `data/csv/*.ledger.jsonl`; CSV thống kê giữ nguyên các cột hiện có.

## 1. Kết nối Drive và khởi tạo môi trường

Code GitHub được clone/pull vào ổ **cục bộ** Colab `/content/arxiv-code` để tránh lỗi `.git/index.lock` trên Drive. Dữ liệu cũ vẫn được giữ nguyên tại `/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data`. Mỗi runtime mới chạy cell này một lần, sau đó cài thư viện; không cần chạy cell clone cũ. Nếu trước đó đã import `src` từ đường dẫn khác, hãy **Runtime → Restart session**.


In [ ]:
# 1. Mount Drive for PERSISTENT DATA, keep all Git operations on Colab local disk.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    DATA_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data")
    BRANCH_NAME = "feat/milestone1-compliance"
    if not (DATA_DIR / "papers.jsonl").is_file():
        raise FileNotFoundError(f"Missing original dataset: {DATA_DIR / 'papers.jsonl'}")

    if not (CODE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch",
            "--branch", BRANCH_NAME,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        current_branch = subprocess.check_output([
            "git", "-C", str(CODE_DIR), "branch", "--show-current",
        ], text=True).strip()
        if current_branch != BRANCH_NAME:
            raise RuntimeError(f"Unexpected branch {current_branch!r}, expected {BRANCH_NAME}")
        subprocess.run([
            "git", "-C", str(CODE_DIR), "pull", "--ff-only", "origin", BRANCH_NAME,
        ], check=True)

    os.environ["ARXIV_DATA_DIR"] = str(DATA_DIR)
    os.chdir(CODE_DIR)
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))
    if "src.config" in sys.modules:
        from src import config
        if Path(config.DEFAULT_DATA_DIR).resolve() != DATA_DIR.resolve():
            raise RuntimeError("src.config was already loaded with another DATA_DIR; restart runtime")
    from src import config
    assert Path(config.DEFAULT_DATA_DIR).resolve() == DATA_DIR.resolve()

    PROJECT_DIR = str(CODE_DIR)
    print("✅ Environment ready")
    print("Code:", CODE_DIR)
    print("Data:", DATA_DIR)
    subprocess.run(["git", "-C", PROJECT_DIR, "log", "-1", "--oneline"], check=True)
except ImportError:
    print("Local environment: ensure ARXIV_DATA_DIR points to your dataset before importing src.")


In [ ]:
# Cài đặt các thư viện cần thiết
%pip install -r requirements.txt


### 1.1. Cấu hình Semantic Scholar API Key (Tùy chọn)

Để cào danh sách trích dẫn (References) với quota cao hơn (100 requests / 5 phút thay vì quota ẩn danh), bạn có thể nạp API Key từ **Google Colab Secrets**:
1. Nhấn vào biểu tượng **Secrets** (🔑) ở thanh công cụ bên trái giao diện Colab.
2. Bấm **"Add new secret"**, đặt tên bí mật là `S2_API_KEY` và dán API Key vào.
3. Bật quyền **Notebook access** cho secret `S2_API_KEY`.

In [ ]:
import os
import requests
from src import config

# Lấy S2_API_KEY từ Google Colab Secrets (hoặc biến môi trường nếu chạy cục bộ)
try:
    from google.colab import userdata
    s2_key = userdata.get('S2_API_KEY')
except (ImportError, Exception):
    s2_key = os.environ.get('S2_API_KEY', '')

if s2_key:
    s2_key = s2_key.strip()
    os.environ["S2_API_KEY"] = s2_key
    config.S2_API_KEY = s2_key
    print(f"✅ Đã nạp S2_API_KEY thành công: {s2_key[:4]}...{s2_key[-4:]} (độ dài: {len(s2_key)})")

    # Kiểm tra tính hợp lệ của key bằng một request thử nghiệm
    try:
        test_url = "https://api.semanticscholar.org/graph/v1/paper/arXiv:2502.13235?fields=title"
        res = requests.get(test_url, headers={"x-api-key": s2_key}, timeout=10)
        if res.status_code == 200:
            print("🎉 Xác thực Semantic Scholar API thành công! (HTTP 200 OK)")
        elif res.status_code in (401, 403):
            print(f"❌ CẢNH BÁO: Key không hợp lệ hoặc không có quyền (HTTP {res.status_code}): {res.text}")
        else:
            print(f"ℹ️ Phản hồi từ Semantic Scholar API: HTTP {res.status_code}")
    except Exception as err:
        print(f"⚠️ Không thể gửi request kiểm tra API: {err}")
else:
    print("⚠️ Chưa tìm thấy S2_API_KEY. Pipeline sẽ chạy ở chế độ ẩn danh (quota thấp hơn).")
    print("💡 Mẹo: Thêm secret 'S2_API_KEY' trong mục Secrets (biểu tượng chìa khóa 🔑) ở thanh bên trái Colab.")


## 2. Khai báo khoảng arXiv-ID cần xử lý

In [ ]:
#@title Cấu hình tham số Pipeline (Khoảng ID, Limit & Tùy chọn Retry) { display-mode: "form" }

#@markdown ### 1. Khoảng arXiv ID và Giới hạn
#@markdown Nhập `start_id` và `end_id` theo định dạng `YYMM.NNNNN` (ví dụ: `2502.13235`).
start_id = "2502.13235"  #@param {type:"string"}
end_id = "2502.13245"    #@param {type:"string"}

#@markdown Giới hạn số bài báo chạy thử nghiệm (đặt `0` để chạy toàn bộ dải ID):
LIMIT = 0                #@param {type:"integer"}

#@markdown ### 2. Tùy chọn Phục hồi & Retry (Recovery Options)
#@markdown - **resume**: Bỏ qua các ID đã hoàn thành hợp lệ (khuyến nghị giữ `True`).
#@markdown - **retry_errors**: Thử lại ngay các ID từng gặp lỗi (lỗi mạng, HTTP 429/5xx).
#@markdown - **retry_not_found**: Thử lại các ID từng trả về HTTP 404 Not Found.
#@markdown - **retry_partial**: Cào bổ sung trích dẫn chưa đầy đủ (chỉ áp dụng Bước 3).
resume = True            #@param {type:"boolean"}
retry_errors = False     #@param {type:"boolean"}
retry_not_found = False  #@param {type:"boolean"}
retry_partial = False    #@param {type:"boolean"}

#@markdown ### 3. Tối ưu Hiệu năng (Performance Optimization)
#@markdown - **parallel_execution**: Chạy đồng thời Bước 2 (Tải mã nguồn) và Bước 3 (Cào trích dẫn Semantic Scholar).
#@markdown - **arxiv_download_delay**: Giãn cách tối thiểu (giây) giữa các request tải mã nguồn arXiv (mặc định 3.0s theo ToS; có thể hạ 1.5s - 2.0s khi test nhanh).
#@markdown - **batch_size_metadata**: Gom cụm truy vấn arXiv Metadata (mặc định 100 ID/request).
#@markdown - **batch_size_references**: Gom cụm truy vấn trích dẫn Semantic Scholar qua batch endpoint (mặc định 100 ID/request, tối đa 500).
parallel_execution = True  #@param {type:"boolean"}
arxiv_download_delay = 3.0  #@param {type:"number"}
batch_size_metadata = 100  #@param {type:"integer"}
batch_size_references = 100  #@param {type:"integer"}

#@markdown ### 4. Tùy chọn Ổ đĩa Cục bộ Siêu tốc (Fast Scratch NVMe Storage cho Colab)
#@markdown - **use_fast_scratch_disk**: Lưu trữ và xử lý trực tiếp trên `/content/data` (ổ NVMe siêu tốc của Colab) thay vì ghi qua Google Drive FUSE (giúp tốc độ I/O nhanh hơn tới 10x-50x và tránh hoàn toàn tình trạng nghẽn I/O hay treo do FUSE). Dữ liệu sẽ tự động đồng bộ (sync) ngược lại vào Google Drive ở cuối phiên chạy.
use_fast_scratch_disk = True  #@param {type:"boolean"}

#@markdown ### 5. An toàn dữ liệu (chưa chạy crawler cũ mặc định)
#@markdown Để FALSE cho đến khi downloader append-only được hoàn thiện và kiểm thử. Audit vẫn chạy bình thường.
ENABLE_LEGACY_CRAWL = False  #@param {type:"boolean"}

# Dùng chung cho cả ba bước. Giữ resume=True để phục hồi lỗi, không tải lại tất cả.
RECOVERY_OPTIONS = {
    "resume": resume,
    "retry_errors": retry_errors,
    "retry_not_found": retry_not_found,
    "retry_partial": retry_partial,
}

from src.utils import parse_arxiv_range
id_list = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    id_list = id_list[:LIMIT]

if not id_list:
    raise ValueError("id_list rỗng: vui lòng kiểm tra lại start_id và end_id")

print(f"-> Tổng số bài báo sẽ xử lý: {len(id_list)}")
print("ID đầu tiên:", id_list[0] if id_list else "None")
print("ID cuối cùng:", id_list[-1] if id_list else "None")
print("Tùy chọn phục hồi (RECOVERY_OPTIONS):", RECOVERY_OPTIONS)
print(f"Chế độ chạy song song (parallel_execution): {parallel_execution}")
print(f"Độ trễ tải mã nguồn arXiv (arxiv_download_delay): {arxiv_download_delay}s")
print(f"Kích thước batch Metadata (batch_size_metadata): {batch_size_metadata}")
print(f"Kích thước batch References (batch_size_references): {batch_size_references}")
print(f"Sử dụng Fast Scratch NVMe (use_fast_scratch_disk): {use_fast_scratch_disk}")


### 2.1. Audit nhẹ — chỉ đọc dữ liệu đã lưu

**Mục đích:** xem metadata trên Drive ghi nhận bao nhiêu paper, tổng số phiên bản và những phiên bản *có khả năng* còn thiếu. Cell **không tải**, không gọi API, không quét/hash `raw/` và không ghi file.

- `AUDIT_ALL_SAVED_PAPERS = True`: kiểm tra toàn bộ paper đã lưu. Đặt `False` nếu chỉ muốn kiểm tra phạm vi `start_id/end_id` trong cell cấu hình.
- `AUDIT_SAMPLE_SIZE = 10`: số ví dụ hiển thị; không giới hạn dữ liệu được kiểm tra.
- `candidate_missing_versions`: số ứng viên cần xem xét **theo metadata/ledger**, không phải số thư mục TeX thực sự thiếu. `legacy_latest_unverified` là phiên bản cuối có trạng thái legacy thành công/PDF-only nhưng chưa xác minh file.

**Cách dùng:** chạy cell cấu hình ID rồi chạy cell 2.1. Đọc báo cáo; tiếp theo chạy cell 2.2 ở chế độ dry-run. **Không** sử dụng danh sách ứng viên này trực tiếp để tải vì downloader phải kiểm tra trạng thái thư mục `vN` trước.


In [ ]:
# =============================================================
# 2.1 | KIEM TRA PHIEN BAN — CHI DOC (KHONG TAI / KHONG GHI)
# =============================================================
# True: xem tat ca paper da luu; False: chi xem id_list da chon.
AUDIT_ALL_SAVED_PAPERS = True
# So vi du can hien thi trong bao cao.
AUDIT_SAMPLE_SIZE = 10

from src import config
from src.scripts.audit_missing_versions import audit_missing_versions

# Luon doc Drive goc; config.DATA_DIR co the da doi sang scratch.
selected_ids = None if AUDIT_ALL_SAVED_PAPERS else id_list
report = audit_missing_versions(
    config.DEFAULT_DATA_DIR,
    ids=selected_ids,
    sample_size=AUDIT_SAMPLE_SIZE,
)
s = report["summary"]

print("\n========== M1 | KIEM TRA PHIEN BAN (READ-ONLY) ==========")
print("Du lieu:", report["data_dir"])
print(f"Paper hop le:                  {s['valid_metadata_papers']:,}/{s['requested_ids']:,}")
print(f"Tong version trong metadata:   {s['expected_versions']:,}")
print(f"Paper co nhieu version:        {s['multi_version_papers']:,}")
print(f"Version cu can xem xet:        {s['earlier_versions_candidates']:,}")
print(f"Version moi nhat thieu/loi:    {s['latest_needs_attention']:,}")
print(f"Ban cu bao success/pdf_only:   {s['legacy_latest_unverified']:,} (chua xac minh file)")
print(f"Tong ung vien can xem xet:    {s['candidate_missing_versions']:,}")
print(f"Paper thieu thong tin version: {s['unknown_version_papers']:,}")

if report["sample_candidates"]:
    print("\nMot so ung vien (CHUA khang dinh thieu file):")
    for row in report["sample_candidates"]:
        print(f"  {row['versioned_id']}: {row['reason']} [{row['legacy_publication_status']}]")

if report["warnings"]:
    print(f"\nCanh bao metadata ({len(report['warnings'])} vi du):")
    for warning in report["warnings"][:5]:
        print(" -", warning)

print("\n✅ Audit hoan tat: khong goi API, khong quet raw/, khong ghi Drive.")


### 2.2. Multi-version downloader — xem trước rồi mới tải thử

Downloader tạo thư mục **mới** `data/raw/<paper>/vN/` và marker `.version-status.json`; lưu lịch sử vào `versions_manifest.jsonl`. Không ghi đè `raw/<paper>/` cũ, `papers.jsonl`, `references.jsonl` hoặc CSV cũ. Thư mục `vN` đã tồn tại nhưng không được xác thực sẽ báo **CONFLICT**.

**Các biến cần biết:**
- `DOWNLOAD_VERSIONS = False`: **DRY-RUN**; chỉ xem kế hoạch, không gọi API hoặc ghi Drive. Đổi thành `True` khi muốn tải thật.
- `VERSION_USE_CONFIG_RANGE = True`: chỉ xem/tải phạm vi `start_id`, `end_id` (và `LIMIT`) ở cell cấu hình. Giữ `True` khi thử tải.
- `VERSION_LIMIT = 3`: **số lượt thử tải tối đa của lần chạy**, không phải 3 paper, cũng không phải giới hạn tổng dataset. Giữ 3 khi thử; nếu chạy lại, resume sẽ bỏ qua bản đã xác minh.

**Cách dùng an toàn:** (1) Dừng các notebook khác đang ghi lên cùng thư mục Drive. (2) Chạy cell 2.2 với `DOWNLOAD_VERSIONS=False`, kiểm tra `to_download`, `conflicts` và `verified_existing`. (3) Đổi thành `True` để thử 3 phiên bản. (4) Xem log `success/pdf_only/not_found/failed` và marker thật trước khi tăng `VERSION_LIMIT`.

**Hiểu đúng phần trăm:** thanh % tính **số lượt thử đã xử lý / số lượt dự kiến trong lần chạy này**, *không* phải % tải thành công và cũng không phải % hoàn thành tất cả phiên bản trên Drive. Ví dụ `2/3 = 66,7%` dù một lượt thất bại. Callback chạy sau khi ghi nhật ký từng lượt. Cell này độc lập với crawler legacy; giữ `ENABLE_LEGACY_CRAWL=False`.

**Resume/Retry trong bản mới:** tải source từ `https://export.arxiv.org/src/<id>vN` (giãn cách ≥ 3 giây). `VERSION_RETRY_NOT_FOUND=False` bỏ qua các version đã ghi HTTP 404; `VERSION_RETRY_ERRORS=False` không thử ngay lỗi đang trong cooldown (60s, 120s, ... đến 1 giờ) và bỏ qua lỗi 400/401/403/410 cho đến khi bật `True`. Những phiên bản tải thành công vẫn dựa trên marker/hash, không dựa vào nhật ký. Cờ `retry_errors`/`retry_not_found` của crawler legacy không tự động ảnh hưởng cell này.


In [ ]:
# =============================================================
# 2.2 | TAI SOURCE THEO PHIEN BAN — MAC DINH CHI XEM TRUOC
# =============================================================
# False = DRY-RUN (khong tai, khong ghi); True = tai that.
DOWNLOAD_VERSIONS = False
# True = chi xu ly start_id/end_id va LIMIT tu cell cau hinh.
VERSION_USE_CONFIG_RANGE = True
# So LAN THU tai toi da trong phien nay. Bat dau bang 3.
# 0 = khong gioi han; KHONG dung khi thu nghiem.
VERSION_LIMIT = 3

# False = bo qua version da tra HTTP 404 trong manifest.
VERSION_RETRY_NOT_FOUND = False
# False = cho cooldown hoac bo qua loi permanent (403/400).
# True = thu lai loi ngay khi da sua nguyen nhan, KHONG bat mac dinh.
VERSION_RETRY_ERRORS = False

from src import config
from src.download_module.version_downloader import run_version_downloads

if DOWNLOAD_VERSIONS and (not VERSION_USE_CONFIG_RANGE or VERSION_LIMIT < 1):
    raise ValueError(
        "De tai an toan: VERSION_USE_CONFIG_RANGE=True va VERSION_LIMIT >= 1."
    )

# Callback duoc goi SAU KHI da ghi nhat ky cua moi lan thu tai.
# % = so luot DA XU LY / so luot DU KIEN trong lan chay nay.
# 'failed' va 'not_found' van tinh la da THU, KHONG phai tai thanh cong.
def show_download_progress(completed, total, event):
    percent = completed * 100 / total if total else 100.0
    filled = round(percent / 5)  # 20 o, moi o = 5%
    bar = "█" * filled + "░" * (20 - filled)
    print(
        f"[{bar}] {percent:5.1f}% | {completed}/{total} luot thu | "
        f"{event['arxiv_id']}/{event['version']} | {event['status']}",
        flush=True,
    )

print("\n========== M1 | VERSION DOWNLOADER ==========")
print("Che do:", "TAI THAT" if DOWNLOAD_VERSIONS else "DRY-RUN")
print("Pham vi:", "id_list tu cell cau hinh" if VERSION_USE_CONFIG_RANGE else "tat ca paper")
print("Gioi han luot thu:", VERSION_LIMIT)

version_result = run_version_downloads(
    config.DEFAULT_DATA_DIR,    # Drive ben vung, khong phai legacy scratch
    ids=id_list if VERSION_USE_CONFIG_RANGE else None,
    max_downloads=VERSION_LIMIT,
    dry_run=not DOWNLOAD_VERSIONS,
    progress_callback=show_download_progress if DOWNLOAD_VERSIONS else None,
    retry_errors=VERSION_RETRY_ERRORS,
    retry_not_found=VERSION_RETRY_NOT_FOUND,
)

plan = version_result["plan"]["summary"]
attempts_planned = plan.get("to_download", 0)
attempts_this_run = min(attempts_planned, VERSION_LIMIT) if VERSION_LIMIT else attempts_planned
summary = version_result["summary"]

print("\n========== KET QUA ==========")
print(f"Version can tai (toan pham vi): {attempts_planned:,}")
print(f"Da duoc xac minh / bo qua:      {plan.get('verified_existing', 0):,}")
print(f"Xung dot can xem xet:           {plan.get('conflicts', 0):,}")
print(f"Da biet 404, tam bo qua:        {plan.get('known_not_found', 0):,}")
print(f"Loi dang cho cooldown:         {plan.get('deferred_retry', 0):,}")
print(f"Loi can xu ly thu cong:        {plan.get('blocked_errors', 0):,}")
print(f"Du kien thu trong luot nay:    {attempts_this_run:,}")

if not DOWNLOAD_VERSIONS:
    print("✅ DRY-RUN: chua gui request, chua ghi file vao Drive.")
else:
    print(f"Da thu:                         {summary.get('network_attempts', 0):,}")
    print(f"Source thanh cong:              {summary.get('success', 0):,}")
    print(f"PDF-only:                       {summary.get('pdf_only', 0):,}")
    print(f"Not found (404):                {summary.get('not_found', 0):,}")
    print(f"That bai:                       {summary.get('failed', 0):,}")
    print(f"Ung vien chua duoc thu:         {summary.get('remaining_candidates', 0):,}")
    print("✅ Hoan tat luot thu. Kiem tra loi truoc khi mo rong pham vi.")


### 2.3. Thống kê phiên bản & đối chiếu source (có thể chạy sau mỗi đợt tải)

Cell này tạo báo cáo **toàn bộ dataset đã lưu** (cả tháng 01 và tháng 02), một dòng cho mỗi cặp `(arxiv_id, version)`. Báo cáo xác minh marker `.version-status.json` và SHA-256 các file `.tex/.bib` của đúng phiên bản; **không quét/không sửa thư mục legacy**.

- `EXPORT_VERSION_CSV = False` (mặc định): chỉ in thống kê, **không ghi Drive**.
- `EXPORT_VERSION_CSV = True`: xuất **duy nhất** file `data/csv/m1_download_versions.csv` bằng ghi tạm rồi thay thế an toàn; các bảng CSV cũ và dữ liệu nguồn vẫn giữ nguyên.
- `verified_success` / `verified_pdf_only`: dữ liệu được xác minh bằng file/marker, không chỉ bởi manifest.
- `pending`: chưa có marker hoặc sự kiện tải của version; **có thể vẫn có source legacy cấp paper**.
- `missing_files`: manifest báo thành công nhưng thư mục version không còn.
- `conflict`: đã có thư mục version nhưng marker/hash không hợp lệ.
- `not_found` và `failed`: trạng thái tải ghi nhận gần nhất; không tính là đã có TeX.

**Thứ tự chạy:** cấu hình Drive/ID → 2.1 → 2.2 → 2.3. Nên chạy 2.3 sau một lô tải; với dataset lớn có thể mất thời gian vì phải kiểm tra nhiều đường dẫn và hash trên Google Drive. File CSV là **snapshot tại thời điểm chạy 2.3**, không tự cập nhật sau lượt download tiếp theo. Không dùng số dòng manifest làm số version hoàn tất.

**Progress bar:** hiển thị phần trăm **version đã xác minh / tổng version dự kiến**. Phần trăm đo tiến độ audit, không phải tỷ lệ source tải thành công; thanh tiến độ cập nhật sau mỗi version đã kiểm tra.


In [ ]:
# =============================================================
# 2.3 | DOI CHIEU TAT CA VERSION VA XUAT CSV THONG KE MOI
# =============================================================
# False = chi xem bao cao (khong ghi); True = tao/lam moi CSV version.
EXPORT_VERSION_CSV = False
AUDIT_VERSION_SAMPLE_SIZE = 10

from tqdm.auto import tqdm
from src import config
from src.scripts.audit_version_downloads import (
    audit_version_downloads,
    export_version_csv,
)

# Luon doc tren Drive ben vung. Khong gioi han theo id_list:
# CSV phai chua du ca 2 thang du ban tai chung o nhieu phien.
with tqdm(desc="Verify versions", unit="version", dynamic_ncols=True) as pbar:
    def on_version_audit_progress(done, total, row):
        if pbar.total != total:
            pbar.total = total
            pbar.refresh()
        pbar.update(done - pbar.n)
        pbar.set_postfix_str(row["status"])

    version_audit = audit_version_downloads(
        config.DEFAULT_DATA_DIR,
        sample_size=AUDIT_VERSION_SAMPLE_SIZE,
        progress_callback=on_version_audit_progress,
    )
counts = version_audit["summary"]

print("\n========== M1 | DOI CHIEU SOURCE THEO VERSION ==========")
print("Paper hop le:", f"{counts['valid_papers']:,}")
print("Version du kien:", f"{counts['expected_versions']:,}")
print("Source TeX da xac minh:", f"{counts['verified_success']:,}")
print("PDF-only da xac minh:", f"{counts['verified_pdf_only']:,}")
print("Khong tim thay (404):", f"{counts['not_found']:,}")
print("Tai that bai:", f"{counts['failed']:,}")
print("Chua xu ly / legacy chua doi chieu:", f"{counts['pending']:,}")
print("Xung dot folder/marker:", f"{counts['conflicts']:,}")
print("Manifest bao xong nhung mat file:", f"{counts['missing_files']:,}")
print("Paper thieu metadata version:", f"{counts['invalid_version_metadata']:,}")
print("Manifest nam ngoai metadata:", f"{counts['orphan_manifest_keys']:,}")

if version_audit["sample_issues"]:
    print("\nMot so van de can xem xet:")
    for issue in version_audit["sample_issues"]:
        print(" -", issue)

if EXPORT_VERSION_CSV:
    path = export_version_csv(version_audit)
    print(f"\n✅ Da xuat {len(version_audit['rows']):,} dong vao {path}")
else:
    print("\n✅ READ-ONLY: Chua tao hay thay doi CSV tren Drive.")


### 2.4. Đối chiếu source legacy với các revision đã xác minh — KHÔNG ghi dữ liệu

Trước đây source được lưu tại `raw/<paper>/` (URL không kèm `vN`); downloader mới lưu tại `raw/<paper>/v1/`, `v2/`... Vì source legacy thiếu thông tin revision chắc chắn nên **không được tự coi là phiên bản cuối**, kể cả giấy tờ có nhiều phiên bản.

**Cell này làm gì?** So sánh **toàn bộ danh sách đường dẫn tương đối, dung lượng và hash SHA-256** của `.tex/.bib` legacy với từng revision `vN` đã có marker hợp lệ. Bỏ qua các thư mục `vN` khi đọc legacy. Không gọi mạng, không sửa hay di chuyển dữ liệu, không thay đổi CSV/manifest, và **không tự bỏ qua lượt tải theo kết quả so sánh**.

- `LEGACY_COMPARE_ALL = False`: chỉ xét `id_list` trong cell cấu hình, nếu muốn xem cả hai tháng đặt thành `True` (đọc từ toàn bộ metadata đã lưu).
- `LEGACY_COMPARE_LIMIT = 20`: chỉ đọc tối đa 20 paper theo thứ tự ID để tránh hash toàn bộ Drive ngoài ý muốn. Đặt `0` khi chủ động muốn kiểm tra toàn bộ phạm vi đã chọn.
- `exact_match`: khớp nguyên bộ source với **một** revision đã xác minh; chứng minh cùng *nội dung*, không chứng minh lịch sử URL tải.
- `ambiguous_match`: khớp nhiều revision khác nhau có cùng nội dung; không đủ để xác định bản gốc.
- `different`: có revision đã xác minh nhưng không khớp; `no_verified_version`: chưa có revision nguồn đủ tin cậy để đối chiếu.
- `untrusted`: có file lạ/symlink hoặc legacy không đủ điều kiện; `no_legacy` và `legacy_pdf_only`: không có TeX legacy phù hợp để so sánh.

**Cách dùng:** chạy 2.4 sau 2.2/2.3 nếu muốn tìm dữ liệu source trùng. Đây chỉ là báo cáo phục vụ quyết định migration tương lai. **Không có thao tác nào ở bước này thay đổi kế hoạch tải hoặc đánh dấu `vN` đã tải.**

**Progress bar:** hiển thị **paper đã đối chiếu / số paper trong phạm vi** (tối đa `LEGACY_COMPARE_LIMIT`). Không phải phần trăm source khớp. Cell vẫn hoàn toàn chỉ đọc.


In [ ]:
# =============================================================
# 2.4 | DOI CHIEU LEGACY VS vN (CHI DOC, KHONG MIGRATE)
# =============================================================
LEGACY_COMPARE_ALL = False
LEGACY_COMPARE_LIMIT = 20  # 0 = quet toan bo pham vi (co the cham)
LEGACY_SAMPLE_SIZE = 10

from tqdm.auto import tqdm
from src import config
from src.scripts.audit_legacy_matches import audit_legacy_matches

with tqdm(desc="Compare legacy", unit="paper", dynamic_ncols=True) as pbar:
    def on_legacy_compare_progress(done, total, row):
        if pbar.total != total:
            pbar.total = total
            pbar.refresh()
        pbar.update(done - pbar.n)
        pbar.set_postfix_str(row["status"])

    legacy_report = audit_legacy_matches(
        config.DEFAULT_DATA_DIR,
        ids=None if LEGACY_COMPARE_ALL else id_list,
        max_papers=LEGACY_COMPARE_LIMIT,
        sample_size=LEGACY_SAMPLE_SIZE,
        progress_callback=on_legacy_compare_progress,
    )
legacy_summary = legacy_report["summary"]
print("\n========== DOI CHIEU LEGACY VA SOURCE vN ==========")
for key, value in legacy_summary.items():
    print(f"{key:30s}: {value:,}")
print("\nVi du doi chieu:")
for row in legacy_report["sample_rows"]:
    print(
        f"{row['arxiv_id']} | {row['status']} | "
        f"version khop: {row['matched_versions'] or 'khong co'} | {row['reason']}"
    )
print("\nLuu y:", legacy_report["warning"])
print("✅ READ-ONLY: khong tai, khong ghi Drive, khong tu dong gan legacy vao vN.")


### 2.5. Metadata index theo từng version — bước tiếp theo (an toàn)

Tạo danh mục từng cặp `(arxiv_id, version)` từ `papers.jsonl` và journal. **Chưa gọi mạng và không sửa metadata cũ.** Bởi crawler hiện lưu chủ yếu ngày cập nhật cuối, những ngày của phiên bản trung gian chưa xác định sẽ giữ trống và có trường `date_source` để nêu nguồn.

- `EXPORT_VERSION_METADATA = False` (mặc định): chỉ xem thống kê, không ghi Drive.
- `EXPORT_VERSION_METADATA = True`: ghi **duy nhất** `data/metadata_versions.jsonl` bằng snapshot nguyên tử, đủ cả hai tháng.
- Các trường `title/authors/abstract` của snapshot hiện tại vẫn ở `papers.jsonl`, **không** bị gán nhầm là nội dung lịch sử của các version trước.
- Đây là **metadata index bước đầu**, không khẳng định đã thu thập đủ lịch sử thay đổi. Cần xác minh/thu thập thêm nguồn revision trước khi dùng các ngày trung gian để phân tích.


In [ ]:
# =============================================================
# 2.5 | VERSION METADATA INDEX — KHONG SUY DOAN NGAY REVISION
# =============================================================
EXPORT_VERSION_METADATA = False
VERSION_METADATA_SAMPLE_SIZE = 10

from src import config
from src.scripts.build_version_metadata import (
    build_version_metadata,
    export_version_metadata,
)

metadata_report = build_version_metadata(
    config.DEFAULT_DATA_DIR, sample_size=VERSION_METADATA_SAMPLE_SIZE,
)
metadata_summary = metadata_report["summary"]
print("\n========== M1 | VERSION METADATA INDEX ==========")
for name, count in metadata_summary.items():
    print(f"{name:29s}: {count:,}")
print("\n", metadata_report["limitations"])
for warning in metadata_report["sample_issues"]:
    print(" -", warning)

if EXPORT_VERSION_METADATA:
    exported = export_version_metadata(metadata_report)
    print(f"✅ Exported {len(metadata_report['rows']):,} rows to {exported}")
else:
    print("✅ READ ONLY: No metadata file was written.")


### 2.6. Raw references — thu thập đầy đủ vị trí trích dẫn (mặc định dry-run)

Crawler legacy chỉ lưu các references có arXiv ID vào `references.jsonl`. Dữ liệu đã bị loại bỏ trong các lượt cào cũ **không thể phục hồi bằng cách xuất lại file đó**: cần truy vấn Semantic Scholar lại, có giới hạn và kiểm tra số trang.

Cell này là collector **mới, độc lập**. Khi bật tải thật, nó lưu từng reference vào `data/references_raw.jsonl`, kể cả trích dẫn không có arXiv ID, bản ghi thiếu `citedPaper`, hoặc references trùng danh tính nhưng ở vị trí khác. Theo dõi resume tại `data/references_raw_ledger.jsonl`; chỉ ghi raw sau khi lấy đủ toàn bộ các trang của một paper. **Không ghi đè** `references.jsonl`, `papers.jsonl`, CSV legacy hoặc nguồn TeX.

- `FETCH_RAW_REFERENCES = False`: chỉ lập kế hoạch, **không gửi request hoặc ghi file**.
- `RAW_REF_USE_CONFIG_RANGE = True`: giới hạn theo `id_list` ở cell cấu hình.
- `RAW_REF_LIMIT = 3`: tối đa **3 paper cần truy vấn mạng** trong lượt chạy. Chỉ tăng sau khi đã test nhỏ.
- `RAW_REF_RETRY_ERRORS = False`, `RAW_REF_RETRY_NOT_FOUND = False`: giữ chính sách resume an toàn, không tự retry các mục lỗi hoặc 404 chưa cần thiết.

**Lưu ý:** Semantic Scholar trả references cấp *paper*, không bảo đảm chúng thuộc revision `v1/v2/...`; mỗi dòng raw được ghi `version = null`. Kết quả `ok` chỉ nói rằng đã nhận trọn các trang API, **không** chứng minh đã trích xuất citation đầy đủ từ mọi bản LaTeX. Trong bản đầu tiên hãy giữ mọi cờ mặc định, sau đó xem kế hoạch rồi mới bật tải thử.


In [ ]:
# =============================================================
# 2.6 | CAPTURE ALL RAW REFERENCES (DEFAULT DRY-RUN)
# =============================================================
FETCH_RAW_REFERENCES = False
RAW_REF_USE_CONFIG_RANGE = True
RAW_REF_LIMIT = 3
RAW_REF_RETRY_ERRORS = False
RAW_REF_RETRY_NOT_FOUND = False

from tqdm.auto import tqdm
from src import config
from src.crawl_references_module.raw_reference_collector import collect_raw_references

if FETCH_RAW_REFERENCES and (not RAW_REF_USE_CONFIG_RANGE or RAW_REF_LIMIT < 1):
    raise ValueError("Safe test requires RAW_REF_USE_CONFIG_RANGE=True and RAW_REF_LIMIT>=1")

with tqdm(desc="Raw references", unit="paper", dynamic_ncols=True) as bar:
    def on_raw_ref_progress(done, total, event):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(event["status"])

    raw_ref_result = collect_raw_references(
        config.DEFAULT_DATA_DIR,
        ids=id_list if RAW_REF_USE_CONFIG_RANGE else [],
        max_papers=RAW_REF_LIMIT,
        dry_run=not FETCH_RAW_REFERENCES,
        retry_errors=RAW_REF_RETRY_ERRORS,
        retry_not_found=RAW_REF_RETRY_NOT_FOUND,
        progress_callback=on_raw_ref_progress if FETCH_RAW_REFERENCES else None,
    )

print("\n========== M1 | RAW REFERENCES ==========")
for label, value in raw_ref_result["summary"].items():
    print(f"{label:28s}: {value:,}")
if not FETCH_RAW_REFERENCES:
    print("✅ DRY-RUN: no Semantic Scholar API calls or Drive changes.")
else:
    for event in raw_ref_result["processed"][:10]:
        print(event["arxiv_id"], event["status"], event.get("raw_refs_total", 0),
              event.get("completeness", ""), event.get("error", "")[:120])


## 3. Khởi động Bộ theo dõi Benchmark Hệ thống (Bảng 4)

In [ ]:
import os
from pathlib import Path
from src import config, utils
from src.scripts.safe_scratch import start_scratch_session

utils.setup_logging("m1_pipeline", config.LOG_DIR / "m1_pipeline.log")
# Mỗi lần thiết lập tạo một session hoàn toàn mới, không tái sử dụng /content/data.
DRIVE_DATA_DIR = Path(config.DEFAULT_DATA_DIR).resolve()
SCRATCH_SESSION = None
if use_fast_scratch_disk and Path("/content").is_dir() and DRIVE_DATA_DIR != Path("/content/data").resolve():
    SCRATCH_SESSION = start_scratch_session(DRIVE_DATA_DIR, scratch_parent="/content")
    config.DATA_DIR = SCRATCH_SESSION.scratch_dir
    print("✅ Safe Scratch: bản làm việc riêng; chưa ghi dữ liệu nào lên Drive.")
else:
    config.DATA_DIR = DRIVE_DATA_DIR
    print(f"Đang sử dụng dữ liệu trực tiếp: {DRIVE_DATA_DIR}")
config.RAW_DIR = config.DATA_DIR / "raw"
RAW_DIR = config.RAW_DIR
CSV_DIR = config.DATA_DIR / "csv"
# Không tạo thư mục trong Drive ở bước setup; crawler sẽ tự quản lý khi được cho phép.
print(f"DATA_DIR: {config.DATA_DIR}")
print("⚠️ Legacy crawler đang bị khóa mặc định. Chỉ thực thi khi bạn chủ động bật ENABLE_LEGACY_CRAWL.")


## 4. Thực thi Toàn bộ Pipeline Thu thập & Đo lường (Bước 1 - Bước 4)

Thực thi 4 bước thu thập theo đúng quy trình và bọc trong khối `try...finally` để đảm bảo Bảng 4 luôn được xuất:
- **Bước 1 (Bảng 1):** Cào Metadata từ arXiv API (`m1_discovery_stats`) với cơ chế gom nhóm (Batching 100 ID/request) siêu tốc.
- **Bước 2 (Bảng 2) & Bước 3 (Bảng 3):** Tải và trích xuất file nguồn TeX (`m1_download_stats`) đồng thời với Cào danh sách trích dẫn từ Semantic Scholar (`m1_reference_stats`) qua cơ chế gom nhóm Batching (`POST /paper/batch`) và đa luồng an toàn.
- **Bước 4 (Bảng 4):** Dừng tracker và lưu báo cáo hiệu năng hệ thống (`m1_system_benchmark`).


In [ ]:
if not ENABLE_LEGACY_CRAWL:
    raise RuntimeError("Legacy crawler is disabled to protect existing Drive data. Run the read-only audit only; await the append-only downloader.")

import json
from pathlib import Path
from src.scripts.safe_scratch import sync_scratch_session
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from src.metadata_module import crawl_papers
from src.download_module.downloader import download_papers_range
from src.crawl_references_module import crawl_references_range
from src.benchmark import SystemBenchmarkTracker, calculate_storage_mb
from src.utils import clean_arxiv_id

print("-> Đang khởi động SystemBenchmarkTracker để theo dõi Bảng 4...")
# Khởi động background resource tracker (đo RAM, Disk và Wall Time)
# disk_sample_interval_sec=0.0: Tắt quét đĩa định kỳ để tránh nghẽn I/O trên Google Drive (đo dung lượng 1 lần ở cuối tại stop)
tracker = SystemBenchmarkTracker(
    storage_dirs=[config.DATA_DIR],
    sample_interval_sec=0.5,
    disk_sample_interval_sec=0.0,
).start()
print("-> Đã kích hoạt SystemBenchmarkTracker để theo dõi Bảng 4 (m1_system_benchmark.csv).")

# Snapshot tập hợp ID đã có kết quả trước khi bắt đầu phiên chạy
def snapshot_completed_ids(csv_dir, stages=None):
    if stages is None:
        stages = ["m1_discovery_stats", "m1_download_stats", "m1_reference_stats"]
    elif isinstance(stages, str):
        stages = [stages]
    seen = set()
    for base in stages:
        ledger_p = csv_dir / f"{base}.ledger.jsonl"
        jsonl_p = csv_dir / f"{base}.jsonl"
        csv_p = csv_dir / f"{base}.csv"
        # Ưu tiên đọc ledger hoặc jsonl (stream 1 lần, tránh parse redundantly cả jsonl và csv)
        if ledger_p.exists():
            try:
                with open(ledger_p, "r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            r = json.loads(line)
                            aid = r.get("arxiv_id")
                            if aid and r.get("status") in {"ok", "pdf_only", "success"}:
                                seen.add(clean_arxiv_id(aid))
                continue
            except Exception:
                pass
        if jsonl_p.exists():
            try:
                with open(jsonl_p, "r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            r = json.loads(line)
                            aid = r.get("arxiv_id")
                            if aid:
                                seen.add(clean_arxiv_id(aid))
                continue
            except Exception:
                pass
        if csv_p.exists():
            try:
                df_temp = pd.read_csv(csv_p, usecols=["arxiv_id"], dtype={"arxiv_id": str})
                seen.update(df_temp["arxiv_id"].dropna().apply(clean_arxiv_id))
            except Exception:
                pass
    return seen

# Snapshot trạng thái trước khi chạy cho từng công đoạn riêng biệt
initial_download_ids = snapshot_completed_ids(CSV_DIR, stages="m1_download_stats")
initial_discovery_ids = snapshot_completed_ids(CSV_DIR, stages="m1_discovery_stats")
initial_reference_ids = snapshot_completed_ids(CSV_DIR, stages="m1_reference_stats")

# Khởi tạo lại biến kết quả trước khối try để tránh rò rỉ từ các cell chạy trước đó trong notebook
df_downloads = None
df_refs = None
stats_papers = {}
try:
    print(f"=== BẮT ĐẦU CÀO METADATA BÀI BÁO (arXiv API - Batch size: {batch_size_metadata}) ===")
    stats_papers = crawl_papers(
        id_list=id_list,
        out_path=config.DATA_DIR / "papers.jsonl",
        csv_path=CSV_DIR / "m1_discovery_stats.csv",
        batch_size=batch_size_metadata,
        **RECOVERY_OPTIONS,
    )
    print("-> Kết quả cào Metadata:", stats_papers)

    max_workers = 2 if parallel_execution else 1
    mode_name = "ĐỒNG THỜI (Parallel - 2 workers)" if parallel_execution else "TUẦN TỰ (Sequential - 1 worker)"
    print(f"\n=== BẮT ĐẦU THỰC THI {mode_name}: TẢI SOURCE (delay: {arxiv_download_delay}s) VÀ CÀO TRÍCH DẪN (Batch S2: {batch_size_references}) ===\n")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_download = executor.submit(
            download_papers_range,
            id_list=id_list,
            raw_dir=RAW_DIR,
            alt_raw_dir=DRIVE_DATA_DIR / "raw" if (use_fast_scratch_disk and DRIVE_DATA_DIR.resolve() != config.DATA_DIR.resolve()) else None,
            csv_dir=CSV_DIR,
            delay=arxiv_download_delay,
            **RECOVERY_OPTIONS,
            show_progress=True,
        )
        future_refs = executor.submit(
            crawl_references_range,
            id_list=id_list,
            out_path=config.DATA_DIR / "references.jsonl",
            csv_path=CSV_DIR / "m1_reference_stats.csv",
            batch_size=batch_size_references,
            **RECOVERY_OPTIONS,
        )

        df_downloads = future_download.result()
        df_refs = future_refs.result()

    print(f"\n-> Đã ghi thống kê tải mã nguồn cho {len(df_downloads)} ID (gồm cả lỗi).")
    print(df_downloads["download_status"].value_counts())
    print(f"\n-> Đã ghi thống kê trích dẫn cho {len(df_refs)} ID (gồm cả lỗi).")
    print(df_refs["ref_status"].value_counts())
finally:
    print("\n=== KẾT THÚC VÀ LƯU BENCHMARK HỆ THỐNG ===")
    # Ghi nhận chính xác số bài cào thực tế trong phiên chạy này (chống nhân dồn & đảm bảo >= 1)
    target_count = len(id_list) if id_list else 1
    target_clean = {clean_arxiv_id(x) for x in id_list}

    # 1. Xác định số bài tải mã nguồn thực tế trong phiên chạy này (session_downloads)
    session_downloads = None
    if df_downloads is not None and not df_downloads.empty:
        df_target = df_downloads[df_downloads["arxiv_id"].apply(clean_arxiv_id).isin(target_clean)]
        if not RECOVERY_OPTIONS.get("resume", True):
            # Khi resume=False, toàn bộ target_ids đều được tải lại trong phiên này
            session_downloads = df_target
        else:
            # Khi resume=True, chỉ lấy các bài mới được tải trong phiên này (chưa có trước đó)
            session_downloads = df_target[~df_target["arxiv_id"].apply(clean_arxiv_id).isin(initial_download_ids)]

    # 2. Xác định các bài mới hoàn thành qua các công đoạn (session_new_ids)
    current_download_ids = snapshot_completed_ids(CSV_DIR, stages="m1_download_stats")
    current_discovery_ids = snapshot_completed_ids(CSV_DIR, stages="m1_discovery_stats")
    current_reference_ids = snapshot_completed_ids(CSV_DIR, stages="m1_reference_stats")

    new_download = (current_download_ids - initial_download_ids) & target_clean
    new_discovery = (current_discovery_ids - initial_discovery_ids) & target_clean
    new_reference = (current_reference_ids - initial_reference_ids) & target_clean
    session_new_ids = new_download | new_discovery | new_reference

    # 3. Gán actual_papers_count theo thứ tự ưu tiên:
    # - Ưu tiên 1: Có bài tải mã nguồn mới trong phiên này (session_downloads)
    if session_downloads is not None and len(session_downloads) > 0:
        actual_papers_count = len(session_downloads)
    # - Ưu tiên 2: Có bài mới hoàn thành ở công đoạn bất kỳ (session_new_ids, ví dụ ngắt giữa chừng hoặc chỉ chạy metadata/refs)
    elif len(session_new_ids) > 0:
        actual_papers_count = len(session_new_ids)
    # - Ưu tiên 3: Dự phòng khi toàn bộ bài đã hoàn thành từ trước (không có bài mới), gán bằng target_count (len(id_list))
    else:
        actual_papers_count = target_count

    # Chốt chặn an toàn: đảm bảo tuyệt đối không bao giờ bằng 0
    actual_papers_count = max(1, int(actual_papers_count))

    # Tính dung lượng thực tế của toàn bộ thư mục dữ liệu (raw, jsonl, csv)
    final_storage_mb = calculate_storage_mb([config.DATA_DIR])

    print(f"-> Số bài cào thực tế trong phiên này: {actual_papers_count}")
    print(f"-> Dung lượng thư mục dữ liệu ({config.DATA_DIR}): {final_storage_mb} MB")
    df_benchmark = tracker.stop(
        total_papers=actual_papers_count,
        csv_path=CSV_DIR / "m1_system_benchmark.csv",
        final_storage_mb=final_storage_mb,
    )
    print("-> Bảng 4: m1_system_benchmark:")
    display(df_benchmark)

    # Đồng bộ an toàn: chặn xung đột, không ghi đè raw/<id> hiện hữu.
    if SCRATCH_SESSION is not None:
        if "future_download" in locals() and not future_download.done():
            raise RuntimeError("Downloader vẫn đang ghi scratch; không thể đồng bộ.")
        sync_result = sync_scratch_session(SCRATCH_SESSION)
        print("✅ Kết quả Safe Scratch:", sync_result)


### Chạy lại và phục hồi các ID bị lỗi

1. Sau khi runtime gián đoạn, thiết lập lại môi trường và chạy các bước với `resume=True`. Dữ liệu cũ được nhận diện tự động; không cần xóa JSONL/CSV hay thư mục source.
2. Nếu lỗi vừa xảy ra, đợi hết thời gian chờ rồi chạy lại ba cell thu thập. Hệ thống chỉ thử những ID chưa hoàn tất và lỗi đã đến hạn; không có tác vụ retry chạy nền.
3. Muốn thử ngay lỗi mạng hoặc sau khi sửa API key/quyền, đổi `RECOVERY_OPTIONS["retry_errors"] = True` trước khi chạy lại ba cell. Cờ này không tải lại kết quả thành công; đặt về `False` cho lần chạy thông thường.
4. Chỉ bật `retry_not_found` hoặc `retry_partial` khi cần kiểm tra lại nhóm tương ứng. `pdf_only` là kết quả đã kiểm tra không có source TeX, không phải lỗi mạng.

Đọc dòng mới nhất của mỗi ID trong `m1_discovery_stats.ledger.jsonl`, `m1_download_stats.ledger.jsonl` và `m1_reference_stats.ledger.jsonl` để xem `status`, `error_type`, `error`, `attempts` và `next_retry_at` (Unix timestamp). `failed` trong CSV vẫn là lỗi, không có nghĩa ID đã tải thành công. Với references, trạng thái `partial` nằm trong ledger; CSV giữ `success` cho truy vấn đã trả kết quả.

Metadata dùng journal để phục hồi khi dừng trước lúc hợp nhất snapshot; source dùng thư mục tạm và backup khi công bố kết quả. Không xóa các file phục hồi khi một lượt đang chạy. Chi tiết xem [hướng dẫn pipeline](README.md#q3-các-id-từng-tải-lỗi-có-được-thử-lại-không).


## 5. Kiểm tra và Trích xuất Toàn bộ 4 Bảng Thống kê Milestone 1

In [ ]:
import pandas as pd

print("=================================================================")
print("TỔNG HỢP CÁC BẢNG DỮ LIỆU MILESTONE 1 (LƯU TẠI data/csv/)")
print("=================================================================")

t1_file = CSV_DIR / "m1_discovery_stats.csv"
t2_file = CSV_DIR / "m1_download_stats.csv"
t3_file = CSV_DIR / "m1_reference_stats.csv"
t4_file = CSV_DIR / "m1_system_benchmark.csv"

if t1_file.exists():
    print("\n--- 1. BẢNG 1: m1_discovery_stats (Khám phá Metadata) ---")
    df1 = pd.read_csv(t1_file, dtype={"arxiv_id": str})
    display(df1.head(10))

if t2_file.exists():
    print("\n--- 2. BẢNG 2: m1_download_stats (Tải nguồn TeX / Bib) ---")
    df2 = pd.read_csv(t2_file, dtype={"arxiv_id": str})
    display(df2.head(10))

if t3_file.exists():
    print("\n--- 3. BẢNG 3: m1_reference_stats (Trích dẫn Semantic Scholar) ---")
    df3 = pd.read_csv(t3_file, dtype={"arxiv_id": str})
    display(df3.head(10))

if t4_file.exists():
    print("\n--- 4. BẢNG 4: m1_system_benchmark (Benchmark CPU/RAM/Disk trên Colab) ---")
    df4 = pd.read_csv(t4_file)
    print("Dòng kết quả lượt chạy mới nhất:")
    display(df4.tail(1))
    if len(df4) > 1:
        print(f"(Lịch sử toàn bộ {len(df4)} lượt chạy trước đó:)")
        display(df4)